# Send an entity with a lifecycle as JSON and read it back

A lifecycle field goes out as its current status's reference: the code, the label and the status's own fields under properties. It comes back from `{"code": ...}`, which selects the declared status of the template, so the status read back is the catalogue's object.

Use the repository environment for this branch. Run cells in order in a fresh kernel; no published-package install is needed.

## Import the types

In [ ]:
!pip install -q aoa-action-machine

In [1]:
from pydantic import Field, ValidationError

from aoa.action_machine.domain import BaseCoreEntity, BaseStateEntity, Lifecycle, value
from aoa.action_machine.domain.base_domain import BaseDomain
from aoa.action_machine.intents.entity import entity


## Declare the model

In [2]:
class ShopDomain(BaseDomain):
    """Group the shop model declarations."""

    name = "shop"
    description = "Shop model"


@entity(description="Order status", domain=ShopDomain)
class OrderStateEntity(BaseStateEntity):
    """Describe each status of an order."""

    receipt_required: bool = Field(default=False, description="A receipt is required")
    DRAFT = value(code="draft", display_name="Draft")
    PAID = value(code="paid", display_name="Payment received", receipt_required=True)


class OrderLifecycle(Lifecycle[OrderStateEntity]):
    """A draft order is paid."""

    _template = (
        Lifecycle().state(OrderStateEntity.DRAFT).to(OrderStateEntity.PAID).initial()
        .state(OrderStateEntity.PAID).final()
    )


@entity(description="Customer order", domain=ShopDomain)
class OrderCoreEntity(BaseCoreEntity):
    """An order and its lifecycle."""

    id: str = Field(description="Order identifier")
    lifecycle: OrderLifecycle = Field(description="Order lifecycle")


## JSON out

In [3]:
print("1) JSON out")
order = OrderCoreEntity(id="ord-1", lifecycle=OrderLifecycle("paid"))
print("   JSON:", order.model_dump(mode="json")["lifecycle"])
print("   Python dump keeps:", type(order.model_dump()["lifecycle"]).__name__)


1) JSON out
   JSON: {'code': 'paid', 'display_name': 'Payment received', 'properties': {'receipt_required': True}}
   Python dump keeps: OrderLifecycle


## JSON in

In [4]:
print("2) JSON in")
back = OrderCoreEntity.model_validate({"id": "ord-1", "lifecycle": {"code": "paid"}})
print("   current state:", back.lifecycle.current_state)
print("   the declared status:", back.lifecycle.current_status is OrderStateEntity.PAID)
print("   equal to what went out:", OrderCoreEntity.model_validate_json(order.model_dump_json()) == order)


2) JSON in
   current state: paid
   the declared status: True
   equal to what went out: True


## An unknown code

In [5]:
print("3) An unknown code")
try:
    OrderCoreEntity.model_validate({"id": "ord-2", "lifecycle": {"code": "shipped"}})
except ValidationError as exc:
    print("   ValidationError:", exc.errors()[0]["msg"])
else:
    raise AssertionError("shipped is not a state of the template")


3) An unknown code
   ValidationError: Value error, State 'shipped' is not defined on OrderLifecycle. Valid states: draft, paid.


## The published schema

In [6]:
schema = OrderCoreEntity.model_json_schema(mode="validation")["properties"]["lifecycle"]
print("4) Schema codes:", schema["properties"]["code"]["enum"])


4) Schema codes: ['draft', 'paid']


The properties written with a status are for the reader; they are not restored. Reading back chooses the declared status by its code, the same object every lifecycle in that state shares.